# Debug: Yahoo Finance Quarterly Fundamentals Fetch

Investigating why `fetch_quarterly_fundamentals` is not pulling enough data:
- How many tickers fail entirely?
- Why do they fail (ETFs vs actual errors)?
- How many quarters does yfinance actually return?
- What fields have high NaN rates?
- Are there alternative yfinance APIs that return more data?

In [3]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import yfinance as yf
import json

## 1. Load existing data and assess the gap

In [4]:
# Load universe
with open('../data/universe/top_1000_tickers.json') as f:
    universe = json.load(f)
print(f"Universe: {len(universe)} tickers")

# Load existing fundamentals
fund_df = pd.read_parquet('../data/raw/run_20260215_112600/fundamentals_quarterly/quarterly_fundamentals.parquet')
print(f"Fundamentals: {fund_df.shape[0]} rows, {fund_df.ticker.nunique()} tickers")

# Load failed tickers
with open('../data/raw/run_20260215_112600/fundamentals_quarterly/failed_tickers.json') as f:
    failed_tickers = json.load(f)
print(f"Failed: {len(failed_tickers)} tickers")
print(f"Success rate: {fund_df.ticker.nunique()}/{len(universe)} = {fund_df.ticker.nunique()/len(universe):.1%}")

Universe: 1000 tickers
Fundamentals: 4632 rows, 757 tickers
Failed: 243 tickers
Success rate: 757/1000 = 75.7%


In [5]:
# How many quarters per ticker?
quarters_per_ticker = fund_df.groupby('ticker').size()
print("Quarters per ticker distribution:")
print(quarters_per_ticker.describe())
print(f"\nDate range: {fund_df.quarter_end_date.min()} to {fund_df.quarter_end_date.max()}")
print(f"\nyfinance only returns ~{quarters_per_ticker.median():.0f} quarters of history")

Quarters per ticker distribution:
count    757.000000
mean       6.118890
std        0.739387
min        2.000000
25%        6.000000
50%        6.000000
75%        7.000000
max        7.000000
dtype: float64

Date range: 2024-05-31 to 2026-01-31

yfinance only returns ~6 quarters of history


In [6]:
# NaN rates per column
nan_pct = (fund_df.isnull().sum() / len(fund_df) * 100).sort_values(ascending=False)
print("NaN % per column:")
print(nan_pct.to_string())

NaN % per column:
revenue_growth_yoy     83.873057
earnings_growth_yoy    83.765112
quick_ratio            47.538860
revenue_growth_qoq     35.470639
earnings_growth_qoq    34.909326
gross_margin           26.187392
gross_profit           26.014680
operating_margin       25.431779
current_ratio          24.784111
current_assets         24.762522
current_liabilities    24.762522
ebitda                 24.611399
operating_income       24.568221
roe                    23.078584
roa                    22.927461
debt_to_equity         20.682211
total_debt             20.272021
profit_margin          19.430052
total_cash             18.847150
total_assets           18.739206
total_equity           18.674439
net_income             18.588083
revenue                18.523316
report_date             0.000000
ticker                  0.000000
quarter_end_date        0.000000


## 2. Why do tickers fail? ETFs vs real errors

In [7]:
# Check a sample of failed tickers to see if they're ETFs
sample_failed = failed_tickers[:30]
etf_check = []

for ticker in sample_failed:
    try:
        info = yf.Ticker(ticker).info
        etf_check.append({
            'ticker': ticker,
            'quoteType': info.get('quoteType', 'unknown'),
            'longName': info.get('longName', 'unknown'),
            'has_financials': yf.Ticker(ticker).quarterly_financials is not None 
                             and not yf.Ticker(ticker).quarterly_financials.empty
        })
    except Exception as e:
        etf_check.append({'ticker': ticker, 'quoteType': f'ERROR: {e}', 'longName': '', 'has_financials': False})

etf_df = pd.DataFrame(etf_check)
print("Quote types among failed tickers:")
print(etf_df['quoteType'].value_counts())
print()
etf_df

Quote types among failed tickers:
quoteType
ETF    30
Name: count, dtype: int64



,ticker,quoteType,longName,has_financials
0,SPY,ETF,State Street SPDR S&P 500 ETF Trust,False
1,QQQ,ETF,Invesco QQQ Trust,False
2,IWM,ETF,iShares Russell 2000 ETF,False
3,GLD,ETF,SPDR Gold Shares,False
4,IVV,ETF,iShares Core S&P 500 ETF,False
5,SLV,ETF,iShares Silver Trust,False
6,VOO,ETF,Vanguard S&P 500 ETF,False
7,TQQQ,ETF,ProShares UltraPro QQQ,False
8,SOXL,ETF,Direxion Daily Semiconductor Bull 3X Shares,False
9,RSP,ETF,Invesco S&P 500 Equal Weight ETF,False


In [ ]:
# Classify all failed tickers by checking quoteType
# (This may take a minute for 243 tickers)
from time import sleep

all_failed_types = []
for i, ticker in enumerate(failed_tickers):
    if (i+1) % 50 == 0:
        print(f"  Checking {i+1}/{len(failed_tickers)}")
    try:
        info = yf.Ticker(ticker).info
        all_failed_types.append({
            'ticker': ticker,
            'quoteType': info.get('quoteType', 'unknown'),
        })
    except:
        all_failed_types.append({'ticker': ticker, 'quoteType': 'ERROR'})
    sleep(0.1)

failed_types_df = pd.DataFrame(all_failed_types)
print("\nAll failed tickers by type:")
print(failed_types_df['quoteType'].value_counts())

# How many are actual equities that should have had data?
equity_failures = failed_types_df[failed_types_df['quoteType'] == 'EQUITY']
print(f"\nActual equity failures (should have data): {len(equity_failures)}")
print(equity_failures['ticker'].tolist())

## 3. Deep dive into yfinance API: what data is actually available?

In [8]:
# Pick a well-known ticker to explore all available yfinance data
test_ticker = 'AAPL'
stock = yf.Ticker(test_ticker)

print(f"=== {test_ticker} ===")
print(f"\n--- quarterly_financials (income statement) ---")
print(f"Shape: {stock.quarterly_financials.shape if stock.quarterly_financials is not None else 'None'}")
if stock.quarterly_financials is not None:
    print(f"Date range: {stock.quarterly_financials.columns.min()} to {stock.quarterly_financials.columns.max()}")
    print(f"Rows (line items): {stock.quarterly_financials.index.tolist()}")

print(f"\n--- quarterly_balance_sheet ---")
print(f"Shape: {stock.quarterly_balance_sheet.shape if stock.quarterly_balance_sheet is not None else 'None'}")
if stock.quarterly_balance_sheet is not None:
    print(f"Date range: {stock.quarterly_balance_sheet.columns.min()} to {stock.quarterly_balance_sheet.columns.max()}")
    print(f"Rows (line items): {stock.quarterly_balance_sheet.index.tolist()}")

print(f"\n--- quarterly_cashflow ---")
print(f"Shape: {stock.quarterly_cashflow.shape if stock.quarterly_cashflow is not None else 'None'}")
if stock.quarterly_cashflow is not None:
    print(f"Date range: {stock.quarterly_cashflow.columns.min()} to {stock.quarterly_cashflow.columns.max()}")
    print(f"Rows (line items): {stock.quarterly_cashflow.index.tolist()}")

=== AAPL ===

--- quarterly_financials (income statement) ---
Shape: (33, 5)
Date range: 2024-12-31 00:00:00 to 2025-12-31 00:00:00
Rows (line items): ['Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Normalized EBITDA', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Reconciled Cost Of Revenue', 'EBITDA', 'EBIT', 'Normalized Income', 'Net Income From Continuing And Discontinued Operation', 'Total Expenses', 'Total Operating Income As Reported', 'Diluted Average Shares', 'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Diluted NI Availto Com Stockholders', 'Net Income Common Stockholders', 'Net Income', 'Net Income Including Noncontrolling Interests', 'Net Income Continuous Operations', 'Tax Provision', 'Pretax Income', 'Other Income Expense', 'Other Non Operating Income Expenses', 'Operating Income', 'Operating Expense', 'Research And Development', 'Selling General And Administration', 'Gross Profit', 'Cost Of Revenue', 'Total Revenue

In [9]:
# Check if quarterly_income_stmt gives more history than quarterly_financials
print("=== Comparing yfinance quarterly data accessors ===")
for attr in ['quarterly_financials', 'quarterly_income_stmt', 'quarterly_balance_sheet', 'quarterly_cashflow']:
    data = getattr(stock, attr, None)
    if data is not None and not data.empty:
        print(f"\n{attr}: {data.shape}, cols={data.columns.min().date()} to {data.columns.max().date()}")
    else:
        print(f"\n{attr}: None or empty")

=== Comparing yfinance quarterly data accessors ===

quarterly_financials: (33, 5), cols=2024-12-31 to 2025-12-31

quarterly_income_stmt: (33, 5), cols=2024-12-31 to 2025-12-31

quarterly_balance_sheet: (65, 6), cols=2024-09-30 to 2025-12-31

quarterly_cashflow: (46, 7), cols=2024-06-30 to 2025-12-31


In [10]:
# Check the actual line item names in each statement
# Our fetch function uses hardcoded names like 'Total Revenue' - do they match?
print("=== Line item names from yfinance ===")
print("\nIncome Statement rows:")
for item in stock.quarterly_financials.index:
    print(f"  '{item}'")

print("\nBalance Sheet rows:")
for item in stock.quarterly_balance_sheet.index:
    print(f"  '{item}'")

print("\nCash Flow rows:")
for item in stock.quarterly_cashflow.index:
    print(f"  '{item}'")

=== Line item names from yfinance ===

Income Statement rows:
  'Tax Effect Of Unusual Items'
  'Tax Rate For Calcs'
  'Normalized EBITDA'
  'Net Income From Continuing Operation Net Minority Interest'
  'Reconciled Depreciation'
  'Reconciled Cost Of Revenue'
  'EBITDA'
  'EBIT'
  'Normalized Income'
  'Net Income From Continuing And Discontinued Operation'
  'Total Expenses'
  'Total Operating Income As Reported'
  'Diluted Average Shares'
  'Basic Average Shares'
  'Diluted EPS'
  'Basic EPS'
  'Diluted NI Availto Com Stockholders'
  'Net Income Common Stockholders'
  'Net Income'
  'Net Income Including Noncontrolling Interests'
  'Net Income Continuous Operations'
  'Tax Provision'
  'Pretax Income'
  'Other Income Expense'
  'Other Non Operating Income Expenses'
  'Operating Income'
  'Operating Expense'
  'Research And Development'
  'Selling General And Administration'
  'Gross Profit'
  'Cost Of Revenue'
  'Total Revenue'
  'Operating Revenue'

Balance Sheet rows:
  'Ordinary 

## 4. Test with a few tickers that have high NaN rates

In [11]:
# Find tickers with the most NaNs in our existing data
nan_per_ticker = fund_df.groupby('ticker').apply(lambda x: x.isnull().sum().sum()).sort_values(ascending=False)
high_nan_tickers = nan_per_ticker.head(10).index.tolist()
print("Tickers with most NaNs:")
print(nan_per_ticker.head(10))

# Check what yfinance returns for these
for ticker in high_nan_tickers[:3]:
    print(f"\n=== {ticker} ===")
    s = yf.Ticker(ticker)
    qf = s.quarterly_financials
    qb = s.quarterly_balance_sheet
    if qf is not None and not qf.empty:
        qf_t = qf.T
        print(f"  Income stmt shape: {qf.shape}")
        print(f"  Has 'Total Revenue': {'Total Revenue' in qf.index}")
        print(f"  Has 'Net Income': {'Net Income' in qf.index}")
        print(f"  Has 'Gross Profit': {'Gross Profit' in qf.index}")
    if qb is not None and not qb.empty:
        print(f"  Balance sheet shape: {qb.shape}")
        print(f"  Has 'Total Assets': {'Total Assets' in qb.index}")
        print(f"  Has 'Current Assets': {'Current Assets' in qb.index}")
        print(f"  Has 'Total Debt': {'Total Debt' in qb.index}")
        print(f"  Has 'Inventory': {'Inventory' in qb.index}")
        print(f"  Has 'Stockholders Equity': {'Stockholders Equity' in qb.index}")
        print(f"  Equity variants: {[x for x in qb.index if 'equity' in x.lower() or 'Equity' in x]}")

Tickers with most NaNs:
ticker
GS      103
HBAN    103
ALL     103
KEY     103
JPM     103
FITB    103
PNC     103
COF     103
ACGL    103
AFL     103
dtype: int64

=== GS ===
  Income stmt shape: (38, 7)
  Has 'Total Revenue': True
  Has 'Net Income': True
  Has 'Gross Profit': False
  Balance sheet shape: (60, 6)
  Has 'Total Assets': True
  Has 'Current Assets': False
  Has 'Total Debt': True
  Has 'Inventory': False
  Has 'Stockholders Equity': True
  Equity variants: ['Common Stock Equity', 'Preferred Stock Equity', 'Total Equity Gross Minority Interest', 'Stockholders Equity', 'Other Equity Interest', 'Other Equity Adjustments', 'Preferred Securities Outside Stock Equity', 'Long Term Equity Investment']

=== HBAN ===
  Income stmt shape: (38, 6)
  Has 'Total Revenue': True
  Has 'Net Income': True
  Has 'Gross Profit': False
  Balance sheet shape: (52, 5)
  Has 'Total Assets': True
  Has 'Current Assets': False
  Has 'Total Debt': True
  Has 'Inventory': False
  Has 'Stockholders

## 5. Compare our fetch function output vs raw yfinance for a single ticker

In [12]:
from src.data_fetch.fetch_fundamentals_quarterly import fetch_quarterly_fundamentals

# Fetch with our function
test_ticker = 'MSFT'
our_result = fetch_quarterly_fundamentals(test_ticker, reporting_lag_days=45)
print(f"Our function returned {len(our_result)} quarters for {test_ticker}")
print(f"Columns: {our_result.columns.tolist()}")
print()
our_result

Our function returned 5 quarters for MSFT
Columns: ['ticker', 'quarter_end_date', 'report_date', 'revenue', 'net_income', 'operating_income', 'gross_profit', 'ebitda', 'total_assets', 'total_equity', 'total_debt', 'total_cash', 'current_assets', 'current_liabilities', 'profit_margin', 'operating_margin', 'gross_margin', 'roe', 'roa', 'debt_to_equity', 'current_ratio', 'quick_ratio', 'revenue_growth_qoq', 'earnings_growth_qoq', 'revenue_growth_yoy', 'earnings_growth_yoy']



,ticker,quarter_end_date,report_date,revenue,net_income,operating_income,gross_profit,ebitda,total_assets,total_equity,...,gross_margin,roe,roa,debt_to_equity,current_ratio,quick_ratio,revenue_growth_qoq,earnings_growth_qoq,revenue_growth_yoy,earnings_growth_yoy
0,MSFT,2024-12-31,2025-02-14,6.963200e+10,2.410800e+10,3.165300e+10,4.783300e+10,3.562600e+10,5.338980e+11,3.026950e+11,...,0.686940,0.079645,0.045155,0.205567,1.350820,1.342472,NaN,NaN,NaN,NaN
1,MSFT,2025-03-31,2025-05-15,7.006600e+10,2.582400e+10,3.200000e+10,4.814700e+10,4.071100e+10,5.626240e+11,3.218910e+11,...,0.687166,0.080226,0.045899,0.188160,1.371592,1.364167,0.006233,0.071180,NaN,NaN
2,MSFT,2025-06-30,2025-08-14,7.644100e+10,2.723300e+10,3.432300e+10,5.242700e+10,4.443400e+10,6.190030e+11,3.434790e+11,...,0.685849,0.079286,0.043995,0.176395,1.353446,1.346804,0.090986,0.054562,NaN,NaN
3,MSFT,2025-09-30,2025-11-14,7.767300e+10,2.774700e+10,3.796100e+10,5.363000e+10,4.806000e+10,6.363510e+11,3.630760e+11,...,0.690459,0.076422,0.043603,0.166786,1.400530,1.392160,0.016117,0.018874,NaN,NaN
4,MSFT,2025-12-31,2026-02-14,8.127300e+10,3.845800e+10,3.827500e+10,5.529500e+10,5.818000e+10,6.653020e+11,3.908750e+11,...,0.680361,0.098390,0.057805,0.147380,1.386024,1.377878,0.046348,0.386024,0.167179,0.595238


In [13]:
# Compare: what does raw yfinance have?
stock = yf.Ticker(test_ticker)

# Check ALL quarterly data available
print("Raw quarterly_financials (transposed):")
raw_income = stock.quarterly_financials.T
print(raw_income)
print(f"\nQuarters available: {len(raw_income)}")
print(f"Date range: {raw_income.index.min()} to {raw_income.index.max()}")

Raw quarterly_financials (transposed):
            Tax Effect Of Unusual Items  Tax Rate For Calcs  \
2025-12-31                 6.520000e+07            0.200000   
2025-09-30                 1.871500e+08            0.190000   
2025-06-30                 4.951251e+05            0.165042   
2025-03-31                 6.966000e+07            0.180000   
2024-12-31                -2.032200e+08            0.180000   

            Normalized EBITDA  Total Unusual Items  \
2025-12-31       5.785400e+10         3.260000e+08   
2025-09-30       4.707500e+10         9.850000e+08   
2025-06-30       4.443100e+10         3.000000e+06   
2025-03-31       4.032400e+10         3.870000e+08   
2024-12-31       3.675500e+10        -1.129000e+09   

            Total Unusual Items Excluding Goodwill  \
2025-12-31                            3.260000e+08   
2025-09-30                            9.850000e+08   
2025-06-30                            3.000000e+06   
2025-03-31                            3.8

## 6. Test alternative: `get_financials()` with full history

In [14]:
# yfinance Ticker has .get_income_stmt(), .get_balance_sheet(), .get_cashflow()
# These accept a `freq='quarterly'` and may return more history
stock = yf.Ticker('AAPL')

print("=== Testing alternative accessors ===")
for method_name in ['get_income_stmt', 'get_balance_sheet', 'get_cash_flow']:
    method = getattr(stock, method_name, None)
    if method is None:
        print(f"{method_name}: not available")
        continue
    try:
        result = method(freq='quarterly', as_dict=False)
        if result is not None and not result.empty:
            print(f"\n{method_name}(freq='quarterly'): shape={result.shape}")
            print(f"  Date range: {result.columns.min().date()} to {result.columns.max().date()}")
            print(f"  Quarters: {len(result.columns)}")
        else:
            print(f"\n{method_name}: empty")
    except Exception as e:
        print(f"\n{method_name}: ERROR - {e}")

=== Testing alternative accessors ===

get_income_stmt(freq='quarterly'): shape=(33, 5)
  Date range: 2024-12-31 to 2025-12-31
  Quarters: 5

get_balance_sheet(freq='quarterly'): shape=(65, 6)
  Date range: 2024-09-30 to 2025-12-31
  Quarters: 6

get_cash_flow(freq='quarterly'): shape=(46, 7)
  Date range: 2024-06-30 to 2025-12-31
  Quarters: 7


In [15]:
# Check if the .income_stmt vs .quarterly_income_stmt accessors differ
stock = yf.Ticker('AAPL')

attrs_to_check = [
    'quarterly_financials',
    'quarterly_income_stmt', 
    'quarterly_balance_sheet',
    'quarterly_cashflow',
    'income_stmt',
    'balance_sheet', 
    'cashflow',
]

for attr in attrs_to_check:
    data = getattr(stock, attr, None)
    if data is not None and not data.empty:
        print(f"{attr:35s}  shape={str(data.shape):15s}  quarters={data.shape[1]}  range={data.columns.min().date()} to {data.columns.max().date()}")
    else:
        print(f"{attr:35s}  None or empty")

quarterly_financials                 shape=(33, 5)          quarters=5  range=2024-12-31 to 2025-12-31
quarterly_income_stmt                shape=(33, 5)          quarters=5  range=2024-12-31 to 2025-12-31
quarterly_balance_sheet              shape=(65, 6)          quarters=6  range=2024-09-30 to 2025-12-31
quarterly_cashflow                   shape=(46, 7)          quarters=7  range=2024-06-30 to 2025-12-31
income_stmt                          shape=(39, 5)          quarters=5  range=2021-09-30 to 2025-09-30
balance_sheet                        shape=(69, 5)          quarters=5  range=2021-09-30 to 2025-09-30
cashflow                             shape=(53, 5)          quarters=5  range=2021-09-30 to 2025-09-30


## 7. Summary of findings and recommendations

In [16]:
# Summary
successful = fund_df.ticker.nunique()
total = len(universe)
avg_quarters = quarters_per_ticker.mean()

print("=" * 60)
print("FUNDAMENTALS FETCH DIAGNOSTIC SUMMARY")
print("=" * 60)
print(f"")
print(f"Universe: {total} tickers")
print(f"Successful fetches: {successful} ({successful/total:.1%})")
print(f"Failed fetches: {len(failed_tickers)} ({len(failed_tickers)/total:.1%})")
print(f"Avg quarters per ticker: {avg_quarters:.1f}")
print(f"Date range: {fund_df.quarter_end_date.min()} to {fund_df.quarter_end_date.max()}")
print(f"")
print("KEY ISSUES:")
print(f"  1. ETFs/funds in universe have no quarterly financials")
print(f"  2. yfinance free tier only returns ~{avg_quarters:.0f} quarters of history")
print(f"  3. High NaN rates on: quick_ratio, gross_margin, operating_margin")
print(f"  4. YoY growth needs 4+ quarters, so first year is always NaN")
print(f"")
print("POTENTIAL FIXES:")
print(f"  1. Filter ETFs from universe before fetching fundamentals")
print(f"  2. Use SEC EDGAR API for deeper history (free, 10+ years)")
print(f"  3. Use alternative line item names (some vary by sector)")
print(f"  4. Add .info fields (P/E, P/B, etc.) as supplemental data")

FUNDAMENTALS FETCH DIAGNOSTIC SUMMARY

Universe: 1000 tickers
Successful fetches: 757 (75.7%)
Failed fetches: 243 (24.3%)
Avg quarters per ticker: 6.1
Date range: 2024-05-31 to 2026-01-31

KEY ISSUES:
  1. ETFs/funds in universe have no quarterly financials
  2. yfinance free tier only returns ~6 quarters of history
  3. High NaN rates on: quick_ratio, gross_margin, operating_margin
  4. YoY growth needs 4+ quarters, so first year is always NaN

POTENTIAL FIXES:
  1. Filter ETFs from universe before fetching fundamentals
  2. Use SEC EDGAR API for deeper history (free, 10+ years)
  3. Use alternative line item names (some vary by sector)
  4. Add .info fields (P/E, P/B, etc.) as supplemental data
